In [0]:
from datetime import datetime, timezone, timedelta
import requests, io, re
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, DoubleType, LongType, NullType

# ── Pakai Unity Catalog Connection ───────────────────────────
# Tidak perlu TENANT_ID, CLIENT_ID, CLIENT_SECRET lagi
CONNECTION_NAME = "sharepoint-enesis-poc"
BASE_URL        = "https://graph.microsoft.com/v1.0"
SITE_ID         = "5031edae-ec62-42cf-9f7a-fb908bc66b79"

# Folder target
FOLDER_PATH = ["99. Reference", "POC Databricks", "2019"]

# Target Silver
TARGET_CATALOG    = "poc_enesis"
TARGET_SCHEMA     = "silver"
TARGET_TABLE_NAME = "sharepoint_python_sellout_2019"
TARGET_TABLE      = f"`{TARGET_CATALOG}`.`{TARGET_SCHEMA}`.`{TARGET_TABLE_NAME}`"

# Hashkey
HASHKEY_COLS     = ["invoice_number", "date_invoice", "produk_id_mi", "cust_id", "salesman_id", "produk_id_distributor", "subdist_id", "kode_pos"]
HASHKEY_COL_NAME = "hashkey"
AUDIT_COLS       = [
    "hashkey", "ingestion_timestamp", "update_timestamp",
    "source_system", "source_file", "ingestion_method"
]

# Filter modified date
RECENT_DAYS = 7

print(f"✅ Config loaded | Today: {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M UTC')}")
print(f"   Connection : {CONNECTION_NAME}")
print(f"   Folder     : {' → '.join(FOLDER_PATH)}")
print(f"   Target     : {TARGET_TABLE}")
print(f"   Threshold  : last {RECENT_DAYS} day(s)")

✅ Config loaded | Today: 2026-04-14 05:48 UTC
   Connection : sharepoint-enesis-poc
   Folder     : 99. Reference → POC Databricks → 2019
   Target     : `poc_enesis`.`silver`.`sharepoint_python_sellout_2019`
   Threshold  : last 7 day(s)


In [0]:
import requests

def get_access_token():
    # ── Ambil tenant_id & client_id dari Unity Catalog Connection ──
    ctx      = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
    host     = ctx.apiUrl().get()
    token_db = ctx.apiToken().get()

    conn_resp = requests.get(
        f"{host}/api/2.1/unity-catalog/connections/{CONNECTION_NAME}",
        headers={"Authorization": f"Bearer {token_db}"}
    )
    conn_resp.raise_for_status()
    options = conn_resp.json().get("options", {})

    tenant_id     = options.get("tenant_id")
    client_id     = options.get("client_id")
    client_secret = dbutils.secrets.get(scope="sharepoint-enesis", key="client_secret")

    resp = requests.post(
        f"https://login.microsoftonline.com/{tenant_id}/oauth2/v2.0/token",
        data={
            "grant_type":    "client_credentials",
            "client_id":     client_id,
            "client_secret": client_secret,
            "scope":         "https://graph.microsoft.com/.default"
        }
    )
    resp.raise_for_status()
    print("✅ Token retrieved via Databricks Secrets + Connection metadata")
    print(f"   tenant_id : {tenant_id}")
    print(f"   client_id : {client_id}")
    print(f"   secret    : [from Databricks Secrets ✅]")
    return resp.json()["access_token"]

access_token = get_access_token()
HEADERS = {"Authorization": f"Bearer {access_token}"}

✅ Token retrieved via Databricks Secrets + Connection metadata
   tenant_id : 18cc527d-44cc-430d-bd18-88c1417e7efb
   client_id : 77d6bb6a-00dc-47f8-b3d7-8d6437ce152e
   secret    : [from Databricks Secrets ✅]


In [0]:
drives_resp = requests.get(f"{BASE_URL}/sites/{SITE_ID}/drives", headers=HEADERS)
drives_resp.raise_for_status()
drives   = drives_resp.json()["value"]

print("Available drives:")
for d in drives:
    print(f"  - '{d['name']}'")

drive    = next((d for d in drives if d["name"] == "Documents"), drives[0])
drive_id = drive["id"]
print(f"\n✅ Drive: '{drive['name']}'")

Available drives:
  - 'Dokumen'

✅ Drive: 'Dokumen'


In [0]:
def navigate_to_folder(drive_id, folder_path):
    current_id   = "root"
    current_path = "root"
    for folder_name in folder_path:
        url = (
            f"{BASE_URL}/drives/{drive_id}/root/children"
            if current_id == "root"
            else f"{BASE_URL}/drives/{drive_id}/items/{current_id}/children"
        )
        items = requests.get(url, headers=HEADERS).json()["value"]
        found = next((i for i in items if "folder" in i and i["name"] == folder_name), None)
        if not found:
            available = [i["name"] for i in items]
            raise FileNotFoundError(f"'{folder_name}' not found. Available: {available}")
        current_id   = found["id"]
        current_path = f"{current_path} → {folder_name}"
        print(f"   ✅ {current_path}")
    return current_id

print("📂 Navigating...")
folder_id = navigate_to_folder(drive_id, FOLDER_PATH)
print(f"✅ Folder found!")

📂 Navigating...
   ✅ root → 99. Reference
   ✅ root → 99. Reference → POC Databricks
   ✅ root → 99. Reference → POC Databricks → 2019
✅ Folder found!


In [0]:
# Ambil semua table yang sudah ada di sharepoint_ingested
existing_tables_df = spark.sql(f"SHOW TABLES IN `{TARGET_CATALOG}`.`{TARGET_SCHEMA}`")
existing_tables    = [row["tableName"] for row in existing_tables_df.collect()]

print(f"✅ Existing tables in {TARGET_CATALOG}.{TARGET_SCHEMA}:")
for t in existing_tables:
    print(f"   - {t}")

✅ Existing tables in poc_enesis.silver:


In [0]:
def resolve_table_name(filename: str, existing_tables: list) -> str:
    """
    Auto-detect nama table dari nama file.
    
    Logic:
    1. Strip extension & clean nama file
    2. Cek exact match ke existing tables
    3. Kalau tidak exact, cek apakah nama file MENGANDUNG nama table
       (contoh: stores_2 → contains 'stores' → match ke table 'stores')
    4. Return None kalau tidak ada match
    """
    # Clean filename
    base = filename.rsplit(".", 1)[0].lower()          # strip .csv
    base_clean = re.sub(r"[^a-z0-9]+", "_", base).strip("_")

    # 1. Exact match
    if base_clean in existing_tables:
        return base_clean

    # 2. Partial match: cek apakah nama file mengandung nama table
    #    Urutkan dari nama terpanjang supaya lebih spesifik match duluan
    sorted_tables = sorted(existing_tables, key=len, reverse=True)
    for table in sorted_tables:
        if table in base_clean:
            return table

    return None  # Tidak ada match



In [0]:
def is_recently_modified(modified_str: str, days: int) -> bool:
    modified_dt = datetime.fromisoformat(modified_str.replace("Z", "+00:00"))
    cutoff      = datetime.now(timezone.utc) - timedelta(days=days)
    return modified_dt >= cutoff

# ── List file dari folder, exclude yang mengandung "test" ────
items_resp = requests.get(
    f"{BASE_URL}/drives/{drive_id}/items/{folder_id}/children",
    headers=HEADERS
)
items_resp.raise_for_status()
all_items = items_resp.json()["value"]

csv_files = [
    i for i in all_items
    if "file" in i
    and i["name"].lower().endswith(".csv")
    and "test" not in i["name"].lower()    # ← exclude file dengan kata "test"
]

today = datetime.now(timezone.utc)

print(f"{'='*70}")
print(f"{'FILE':<30} {'MODIFIED':<15} {'DAYS AGO':<10} {'ACTION'}")
print(f"{'='*70}")

recent_files  = []
skipped_files = []

for f in csv_files:
    modified_str = f.get("lastModifiedDateTime", "")
    modified_dt  = datetime.fromisoformat(modified_str.replace("Z", "+00:00"))
    days_ago     = (today - modified_dt).days
    hours_ago    = int((today - modified_dt).total_seconds() / 3600)
    is_recent    = is_recently_modified(modified_str, RECENT_DAYS)
    time_label   = f"{hours_ago}h ago" if days_ago == 0 else f"{days_ago}d ago"

    if is_recent:
        action = "✅ MERGE"
        recent_files.append(f)
    else:
        action = "⏭️  SKIP"
        skipped_files.append(f)

    print(f"{f['name']:<30} {modified_str[:10]:<15} {time_label:<10} {action}")

print(f"{'='*70}")
print(f"\n📊 Summary: {len(recent_files)} file(s) to merge | {len(skipped_files)} file(s) skipped")

FILE                           MODIFIED        DAYS AGO   ACTION
2019 - des.csv                 2026-04-07      6d ago     ✅ MERGE
2019 - okt sd nov.csv          2026-04-07      6d ago     ✅ MERGE

📊 Summary: 2 file(s) to merge | 0 file(s) skipped


In [0]:
# ============================================================
# CELL 7 — MERGE FUNCTION (hashkey-based, dengan timestamps)
# ============================================================

from pyspark.sql.types import NullType

def auto_cast_columns(sdf):
    null_cols = [f.name for f in sdf.schema.fields if isinstance(f.dataType, NullType)]
    for col in null_cols:
        sdf = sdf.withColumn(col, F.lit(None).cast(StringType()))

    for col_name in sdf.columns:
        if col_name in AUDIT_COLS:
            continue
        col_lower = col_name.lower()
        if "date" in col_lower and "update" not in col_lower:
            sdf = sdf.withColumn(col_name, F.coalesce(
                F.expr(f"try_to_date(`{col_name}`, 'MM/dd/yyyy')"),
                F.expr(f"try_to_date(`{col_name}`, 'M/d/yyyy')"),
                F.expr(f"try_to_date(`{col_name}`, 'M/dd/yyyy')"),
                F.expr(f"try_to_date(`{col_name}`, 'yyyy-MM-dd')"),
            ))
        elif any(k in col_lower for k in ["qty", "value", "price", "disc", "nett"]):
            sdf = sdf.withColumn(col_name, F.col(col_name).cast(DoubleType()))
        elif col_lower.endswith("_sap") or col_lower in ["subdist_id", "subdist_id_sap"]:
            sdf = sdf.withColumn(col_name, F.col(col_name).cast(LongType()))
        else:
            sdf = sdf.withColumn(col_name, F.col(col_name).cast(StringType()))
    return sdf


def generate_hashkey(sdf):
    concat_expr = F.concat_ws(
        "|",
        *[F.coalesce(F.col(c).cast("string"), F.lit("")) for c in HASHKEY_COLS]
    )
    return sdf.withColumn(HASHKEY_COL_NAME, F.md5(concat_expr))


def merge_file_to_table(file_item: dict, drive_id: str):
    file_name = file_item["name"]
    file_id   = file_item["id"]
    modified  = file_item.get("lastModifiedDateTime", "")[:19]

    print(f"\n{'='*60}")
    print(f"📥 File     : {file_name}  (modified: {modified})")
    print(f"🎯 Target   : {TARGET_TABLE}")

    # ── Download CSV ─────────────────────────────────────────
    dl_resp = requests.get(
        f"{BASE_URL}/drives/{drive_id}/items/{file_id}/content",
        headers=HEADERS
    )
    dl_resp.raise_for_status()

    pdf = pd.read_csv(io.BytesIO(dl_resp.content), dtype=str, low_memory=False)
    pdf = pdf.where(pdf.notna(), None)

    incoming_sdf = spark.createDataFrame(pdf)
    print(f"   Existing rows : {incoming_sdf.count():,}")

    # ── Cast + Hashkey ────────────────────────────────────────
    incoming_sdf = auto_cast_columns(incoming_sdf)
    incoming_sdf = generate_hashkey(incoming_sdf)

    # ── Audit columns (source-level) ─────────────────────────
    incoming_sdf = incoming_sdf \
        .withColumn("source_system",    F.lit("sharepoint")) \
        .withColumn("source_file",      F.lit(file_name)) \
        .withColumn("ingestion_method", F.lit("python_graphapi"))

    incoming_count = incoming_sdf.count()
    incoming_sdf.createOrReplaceTempView("incoming_data")

    table_exists = spark.catalog.tableExists(
        f"{TARGET_CATALOG}.{TARGET_SCHEMA}.{TARGET_TABLE_NAME}"
    )

    if not table_exists:
        # ── FIRST LOAD: INSERT semua ──────────────────────────
        first_df = incoming_sdf \
            .withColumn("ingestion_timestamp", F.current_timestamp()) \
            .withColumn("update_timestamp",    F.current_timestamp())

        first_df.write \
            .format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(TARGET_TABLE)

        total = spark.table(TARGET_TABLE).count()
        print(f"✅ CREATED — {total:,} rows inserted")
        print(f"   ingestion_timestamp = update_timestamp = now()")
        return {"file": file_name, "action": "CREATE",
                "inserted": total, "updated": 0,
                "total_after": total, "status": "SUCCESS"}

    else:
        # ── MERGE ─────────────────────────────────────────────
        # Kolom bisnis = semua kolom kecuali audit
        business_cols = [
            c for c in incoming_sdf.columns
            if c not in AUDIT_COLS + [HASHKEY_COL_NAME]
        ]

        # Change detection — cek semua kolom bisnis
        change_check = " OR ".join([
            f"(target.{c} IS DISTINCT FROM source.{c})"
            for c in business_cols
        ])

        # SET clause — exclude ingestion_timestamp (freeze!)
        set_cols = [
            c for c in incoming_sdf.columns
            if c not in ["ingestion_timestamp", "update_timestamp", HASHKEY_COL_NAME]
        ]
        set_clause = ",\n                    ".join(
            [f"target.{c} = source.{c}" for c in set_cols]
        )

        spark.sql(f"""
            MERGE INTO {TARGET_TABLE} AS target
            USING incoming_data AS source
            ON target.{HASHKEY_COL_NAME} = source.{HASHKEY_COL_NAME}

            -- Data berubah → UPDATE bisnis + update_timestamp = now()
            -- ingestion_timestamp TIDAK BERUBAH ✅
            WHEN MATCHED AND ({change_check}) THEN
                UPDATE SET
                    {set_clause},
                    target.update_timestamp = current_timestamp()

            -- Data baru → INSERT semua + ingestion_timestamp = update_timestamp = now()
            WHEN NOT MATCHED THEN
                INSERT ({", ".join(incoming_sdf.columns)},
                        ingestion_timestamp, update_timestamp)
                VALUES ({", ".join([f"source.{c}" for c in incoming_sdf.columns])},
                        current_timestamp(), current_timestamp())
        """)

        total_after = spark.table(TARGET_TABLE).count()

        # Count inserted vs updated
        existing_hashkeys = spark.table(TARGET_TABLE).select(HASHKEY_COL_NAME)
        new_rows    = incoming_sdf.join(existing_hashkeys, on=HASHKEY_COL_NAME, how="left_anti").count()
        update_rows = incoming_count - new_rows

        print(f"✅ MERGED:")
        print(f"   Inserted : {new_rows:,} new rows")
        print(f"   Updated  : {update_rows:,} rows")
        print(f"   Total    : {total_after:,} rows in silver")

        return {"file": file_name, "action": "MERGE",
                "inserted": new_rows, "updated": update_rows,
                "total_after": total_after, "status": "SUCCESS"}

print("✅ merge_file_to_table ready")

✅ merge_file_to_table ready


In [0]:
# ============================================================
# CELL 8 — RUN MERGE
# ============================================================

spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{TARGET_CATALOG}`.`{TARGET_SCHEMA}`")

results = []

if not recent_files:
    print("⚠️  Tidak ada file yang recently modified. Tidak ada yang di-merge.")
else:
    print(f"🚀 Starting merge for {len(recent_files)} recent file(s)...\n")
    for file_item in recent_files:
        try:
            result = merge_file_to_table(file_item, drive_id)  # ← hapus existing_tables
            results.append(result)
        except Exception as e:
            print(f"❌ FAILED: {file_item['name']} → {e}")
            import traceback
            traceback.print_exc()
            results.append({
                "file": file_item["name"], "action": "FAILED",
                "inserted": 0, "updated": 0,
                "total_after": 0, "status": str(e)[:200]
            })

🚀 Starting merge for 2 recent file(s)...


📥 File     : 2019 - des.csv  (modified: 2026-04-07T09:19:38)
🎯 Target   : `poc_enesis`.`silver`.`sharepoint_python_sellout_2019`
   Existing rows : 614,619
✅ CREATED — 614,619 rows inserted
   ingestion_timestamp = update_timestamp = now()

📥 File     : 2019 - okt sd nov.csv  (modified: 2026-04-07T09:19:38)
🎯 Target   : `poc_enesis`.`silver`.`sharepoint_python_sellout_2019`
   Existing rows : 804,337
✅ MERGED:
   Inserted : 0 new rows
   Updated  : 804,337 rows
   Total    : 1,418,956 rows in silver


In [0]:
print("\n" + "="*60)
print("📊 MERGE SUMMARY")
print("="*60)

if results:
    display(spark.createDataFrame(pd.DataFrame(results)))
else:
    print("Tidak ada file yang diproses.")

# Skipped files
if skipped_files:
    print(f"\n⏭️  SKIPPED ({len(skipped_files)} files — tidak ada perubahan recent):")
    for f in skipped_files:
        modified_dt = datetime.fromisoformat(
            f["lastModifiedDateTime"].replace("Z", "+00:00")
        )
        days_ago = (today - modified_dt).days
        print(f"   - {f['name']:30s} ({days_ago} days ago)")


📊 MERGE SUMMARY


file,action,inserted,updated,total_after,status
2019 - des.csv,CREATE,614619,0,614619,SUCCESS
2019 - okt sd nov.csv,MERGE,0,804337,1418956,SUCCESS


In [0]:
# ── INSERT RUN LOG ────────────────────────────────────────────
import uuid

total_inserted = sum(r.get("inserted", 0) for r in results)
total_updated  = sum(r.get("updated", 0) for r in results)
total_failed   = sum(1 for r in results if r.get("status") != "SUCCESS")

status  = "FAILED" if total_failed > 0 else "SUCCESS"
msg     = f"{len(results)} file(s) processed | {total_inserted} inserted | {total_updated} updated"
if total_failed > 0:
    msg += f" | {total_failed} file(s) FAILED"

log_df = spark.createDataFrame([{
    "run_id"     : str(uuid.uuid4())[:8],
    "schema_name": "GOLD",
    "table_name" : "sharepoint_python_sellout_2019",
    "status"     : status,
    "message"    : msg,
    "rows_in"    : total_inserted + total_updated,
    "rows_out"   : total_inserted,
    "started_at" : started,        # pakai variabel started yang sudah ada di cell sebelumnya
    "ended_at"   : datetime.utcnow()
}])

log_df.write.format("delta").mode("append").saveAsTable("poc_enesis.gold.run_log")
print(f"\n✅ Run log inserted → poc_enesis.gold.run_log | status: {status}")